## This is the code to Analyze the Neighbour Influence 1 Experiment

By default, if you play this file directly, it will generate the radius vs Average influence difference plot with respect to our experiment result.

**The idea of this analysis code is that: We obtain the influence list from the estimation code, and we obtain the neighbour distance by simply constructing the exact same dataset here. Then the distance can be computed by pairwise Euclidean distance. Finally, we map the neighbour distance with the influence difference.**

**Guideline**:  
Construct the exactly same dataset as the influence list that you wish to explore -> Read in the corresponding Influence lists -> Compute the pairwise Euclidean distance between each training sample -> Split the neighbour distance into 4 radius ranges -> Compute the average influence scores difference between neighbours -> Turn the results into the corresponding plot

**Format**:  
**Input** The Influence lists that you read in and the corresponding dataset.  
**Output**  The Radius vs Average influence difference Plot 

IF you want to test on other influence lists, remember, you need to change both the read_csv part and the dataset to the new data. Always make sure the dataset you use for influence estimation is the same here.

In [42]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [43]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [44]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [45]:
import random
from keras.optimizers import SGD

In [46]:
from sklearn.datasets import make_classification
from sklearn.datasets import make_blobs

In [47]:
import seaborn as sns
import matplotlib.pyplot as plt

1. We construct the exactly same dataset as the one used to generate influence here. The reason is that we need to examine the neighbour distance here and then compute the influence difference across neighbours. We won't give detailed explanation to the dataset again here.

In [48]:
train_pool = 16000
test_size = 500
train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]

In [49]:
df = pd.read_csv("diamonds.csv")

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0           60.599998            60.0        6

In [50]:
median_price = df["price"].median()
df["label"] = (df["price"] > median_price).astype(int)
df = df.drop(columns=['price'])

In [51]:
df['id'] = np.arange(1, len(df) + 1)
print(df)

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0           60.599998            60.0        6

In [52]:
df['label'].value_counts()

label
0    26985
1    26955
Name: count, dtype: int64

In [53]:
exact_size = 16500

In [54]:
cur_ratio = ratios[4]
print(cur_ratio)

(5, 5)


In [55]:
major, minor = cur_ratio

In [56]:
df0 = df[df.label == 0]  
df1 = df[df.label == 1] 

In [57]:
t0 = int(exact_size * major / (major + minor))
t1 = exact_size - t0 
print(t0,t1)

8250 8250


In [58]:
s0 = df0.sample(n=t0, random_state=seed)
s1 = df1.sample(n=t1, random_state=seed)

In [59]:
df = pd.concat([s0, s1], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [60]:
print(df)
print(df["label"].value_counts())

       features/carat  features/clarity  features/color  features/cut  \
0                0.56                 5               3             2   
1                0.33                 7               3             4   
2                1.19                 2               5             3   
3                1.35                 2               6             2   
4                1.02                 2               2             2   
...               ...               ...             ...           ...   
16495            1.34                 7               1             3   
16496            1.01                 1               5             0   
16497            0.38                 2               6             4   
16498            0.38                 2               3             4   
16499            1.01                 1               1             3   

       features/depth  features/table  features/x  features/y  features/z  \
0           61.400002            57.0        5

In [61]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [62]:
print(df_train_pool.head())
print(df_test.head())

   features/carat  features/clarity  features/color  features/cut  \
0            0.56                 5               3             2   
1            0.33                 7               3             4   
2            1.19                 2               5             3   
3            1.35                 2               6             2   
4            1.02                 2               2             2   

   features/depth  features/table  features/x  features/y  features/z  label  \
0       61.400002            57.0        5.26        5.32        3.25      0   
1       61.700001            55.0        4.47        4.45        2.75      0   
2       62.200001            58.0        6.73        6.77        4.20      1   
3       61.099998            61.0        7.10        7.13        4.35      1   
4       59.799999            58.0        6.49        6.55        3.90      1   

      id  
0  16508  
1   1783  
2  35552  
3  44044  
4  33617  
   features/carat  features/clarity  f

In [63]:
nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]

In [64]:
train_df = nested_train_dfs[7]

In [65]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[5.6000e-01 5.0000e+00 3.0000e+00 ... 5.3200e+00 3.2500e+00 1.6508e-06]
 [3.3000e-01 7.0000e+00 3.0000e+00 ... 4.4500e+00 2.7500e+00 1.7830e-07]
 [1.1900e+00 2.0000e+00 5.0000e+00 ... 6.7700e+00 4.2000e+00 3.5552e-06]
 ...
 [3.0000e-01 2.0000e+00 1.0000e+00 ... 4.2800e+00 2.7200e+00 5.3008e-06]
 [7.0000e-01 3.0000e+00 5.0000e+00 ... 5.7200e+00 3.5500e+00 3.9702e-06]
 [3.8000e-01 5.0000e+00 3.0000e+00 ... 4.6700e+00 2.9000e+00 3.5662e-06]]


In [66]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[7.0000e-01 3.0000e+00 2.0000e+00 ... 5.7200e+00 3.5400e+00 1.2530e-06]
 [9.1000e-01 1.0000e+00 2.0000e+00 ... 6.2300e+00 3.8200e+00 2.6582e-06]
 [1.0000e+00 5.0000e+00 1.0000e+00 ... 6.4200e+00 3.9700e+00 1.7877e-06]
 ...
 [3.8000e-01 2.0000e+00 6.0000e+00 ... 4.7000e+00 2.9000e+00 3.1090e-07]
 [3.8000e-01 2.0000e+00 3.0000e+00 ... 4.6800e+00 2.8600e+00 3.8341e-06]
 [1.0100e+00 1.0000e+00 1.0000e+00 ... 6.3900e+00 3.9900e+00 2.5028e-06]]


2. After we construct the dataset, we read in the corresponding ranked influence list. **The one you generated in the estimation code using exactly the same dataset as above.** We need the score to compute the influence difference between neighbours.

In [67]:
print(X_train.shape)

(8000, 10)


In [68]:
ranked_df = pd.read_csv("TC_Train_Set_Diamonds_Neighbor.csv")

In [69]:
ranked_sorted = ranked_df.sort_values("Train_ID")
print(ranked_sorted)

      Train_ID         Score
117          7  6.716408e-05
6920        13 -1.803757e-07
794         32  5.706242e-06
7746        45 -3.099641e-05
6968        64 -2.265069e-07
...        ...           ...
1760     53914  5.820241e-07
4235     53916 -2.322466e-13
2605     53922  1.441747e-07
4145     53933 -6.493152e-14
6960     53937 -2.172241e-07

[8000 rows x 2 columns]


In [70]:
print(X_train[:,-1:])

[[1.6508e-06]
 [1.7830e-07]
 [3.5552e-06]
 ...
 [5.3008e-06]
 [3.9702e-06]
 [3.5662e-06]]


In [71]:
infl_train = ranked_sorted["Score"].to_numpy(dtype=np.float64)
print(infl_train.shape)

(8000,)


3. The Pairwise distance between each training sample is then computed to find out the close neighbours distance. Radius is determined here to cover only the real neighbour initially.

In [72]:
from sklearn.neighbors import BallTree
from sklearn.metrics import pairwise_distances

In [73]:
D = pairwise_distances(X_train, metric="euclidean")
tri = D[np.triu_indices_from(D, k=1)]

In [74]:
r_cap = tri.max()
radius_ratios = [0.01, 0.05, 0.10, 0.20, 0.30]
radii = [r * r_cap for r in radius_ratios]

In [75]:
# r_cap = tri.max()
# tri_cap = tri[tri <= r_cap]
# radii = np.quantile(tri_cap, [0.20, 0.40, 0.60, 0.80, 1.00])
# print("Shell bounds (≤ max/2):", np.round(radii, 4))

In [76]:
# radii = [5,6,7,8,9]

In [77]:
tree = BallTree(X_train, metric="euclidean")

4. Then we find the average influence difference between neighbours and then increase the radius. The corresponding plot is then generated.

In [78]:
records = []
prev_r = 0.0
for r in radii:
    inds_list, dists_list = tree.query_radius(X_train, r=r, return_distance=True, sort_results=False)
    diffs = []
    covered = 0
    for i, (nbrs, dists) in enumerate(zip(inds_list, dists_list)):
        mask = (dists > prev_r) & (dists <= r)
        nbrs = nbrs[mask]
        nbrs = nbrs[nbrs != i]
        if nbrs.size > 0:
            diffs.append(np.mean(np.abs(infl_train[i] - infl_train[nbrs])))
            covered += 1

    avg_diff = np.mean(diffs) if diffs else np.nan
    coverage = covered / len(X_train)
    records.append({"radius": r, "shell_from": prev_r, "avg_inf_diff": avg_diff, "coverage": coverage})

    prev_r = r


In [79]:
summary = pd.DataFrame(records)
print(summary)

     radius  shell_from  avg_inf_diff  coverage
0  0.254807    0.000000      0.000012  0.247250
1  1.274035    0.254807      0.000014  0.903250
2  2.548071    1.274035      0.000014  0.996250
3  5.096141    2.548071      0.000014  0.999625
4  7.644212    5.096141      0.000014  0.999875


In [80]:
summary.to_csv("summary_tc_dia.csv",index = False)